# Packages import

In [ ]:
import os
import re
import json
import requests
from dateutil.parser import isoparse
from datetime import timezone
from flask import Flask
from bs4 import BeautifulSoup

## News scraper

1. Provide URL address of article webpage

In [2]:
article_code = "2026/06/cybersecurity-stars-awards-2026-winners.html"
url = f"https://thehackernews.com/{article_code}"

2. Send the request to provided URL address

In [3]:
response = requests.get(url)
print(response.status_code)

200


3. If status code is OK, fetch article name

In [4]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [5]:
article_name = page_dom.select_one("#app > div > h1 > a").get_text()
print(article_name)

Cybersecurity Stars Awards 2026: Winners Announced Across 95 Categories


4. If status code is OK, fetch article from requested webpage

In [6]:
article = page_dom.select("#app > div")
print(type(article))
print(len(article))

<class 'bs4.element.ResultSet'>
1


5. For all fetched articles, parse them to extract relevant data

In [7]:
whole_article = []
for ar in article:
    single_article = {
        'title': ar.select_one('h1 > a').text.strip(),
        'author': ar.select_one("div.postmeta > span.p-author > span:nth-child(2)").text.strip(),
        'pub_date': isoparse(ar.select_one('meta[itemprop="datePublished"]')["content"]).astimezone(timezone.utc).isoformat(),
        'mod_date': isoparse(ar.select_one('meta[itemprop="dateModified"]')["content"]).astimezone(timezone.utc).isoformat(),
        'tags': ar.select_one("div.postmeta > span.p-tags").text.strip(), 
        'body': ar.select_one("#articlebody").text.strip()
    }
    whole_article.append(single_article)

6. Save obtained article

In [8]:
if not os.path.exists("./articles"):
    os.mkdir("./articles")

In [12]:
safe_name = re.sub(r'[\\/*?:"<>|]', '', article_name).strip().replace(' ', '_')
fname = os.path.join("articles", f"{safe_name}.json")
json_text = json.dumps(whole_article, indent=4, ensure_ascii=False)
with open(fname, "w", encoding="utf-8") as jf:
    jf.write(json_text)

# Flask integration

7. Create a Flask application